In [ ]:
import json
from collections import defaultdict
import sys
import matplotlib.pyplot as plt
import numpy as np
import os

print(os.getcwd())

def get_data(dataset, target_model, mode):
    
    data_path = f"../results/evaluation/{target_model}/{mode}/{dataset}/samples.jsonl"
    records = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))

    return records[:100]

def get_baseline_data(dataset, target_model, mode):
    data_path = f"../results/{mode}/{target_model}/{dataset}/samples.jsonl"
    records = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))

    return records[:100]

def get_raw_cot(baseline_data, data):
    try:
        baseline_cots = [sample["ground_truth_cot"] for sample in baseline_data]
    except:
        baseline_cots = [
        sample["all_scratchpads"][0].get("cot")
        if sample.get("all_scratchpads")
        else None
        for sample in baseline_data
    ]
    tool_cots = [
    sample["all_scratchpads"][0].get("cot")
    if sample.get("all_scratchpads")
    else None
    for sample in data
]
    return baseline_cots, tool_cots

dataset = "OpenThoughts"
target_model = "deepseek-v4-flash"

baseline_data = get_baseline_data(dataset, target_model, "baseline_v0")
data = get_data(dataset, target_model, "length_lexical")
manual_data = get_data(dataset, target_model, "manual_injection")

baseline_cots, tool_cots = get_raw_cot(baseline_data, data)

/Users/yitingqu/Documents/projects/CoTRecon/analysis


In [ ]:
import unicodedata
import difflib
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")




def token_em(reconstructed: str, ground_truth: str, *, min_run: int = 1) -> float:
    def _normalize_text(text: str) -> str:
        s = unicodedata.normalize("NFKC", text)
        s = " ".join(s.split())
        return s

    a = _ENC.encode(_normalize_text(reconstructed))
    b = _ENC.encode(_normalize_text(ground_truth))

    if not b:
        return 0.0

    sm = difflib.SequenceMatcher(a=a, b=b, autojunk=False)
    matched = 0
    for tag, i1, i2, _, _ in sm.get_opcodes():
        if tag == "equal":
            run = i2 - i1
            if run >= min_run:
                matched += run
    return matched / len(b)

In [16]:
dataset = "OpenThoughts"
target_model = "qwen3.5-plus"
data = get_data(dataset, target_model, "length_lexical")
target_cot = []
extracted_cot = []
for sample in data:
    success_turn = sample["success_turn"]
    all_scratchpads = sample["all_scratchpads"]
    try:
        target_cot = all_scratchpads[0]["cot"]
        extracted_cot = all_scratchpads[success_turn]["scratchpad_content"]
    except:
        continue
    length_error = all_scratchpads[success_turn]["length_error"]
    token_f1, rouge_l = all_scratchpads[success_turn]["token_f1"], all_scratchpads[success_turn]["rouge_l"]
    emr = token_em(extracted_cot, target_cot)
    print("length_error:", length_error, "token_f1:", token_f1, "rouge_l:", rouge_l)
    print("Exact Match Rate:", emr)
    print("===="*20)

length_error: 0.368 token_f1: 0.494 rouge_l: 0.292
Exact Match Rate: 0.2437137330754352
length_error: 0.545 token_f1: 0.543 rouge_l: 0.452
Exact Match Rate: 0.32916145181476847
length_error: 7.495 token_f1: 0.104 rouge_l: 0.12
Exact Match Rate: 0.23076923076923078
length_error: 4.346 token_f1: 0.26 rouge_l: 0.178
Exact Match Rate: 0.3827392120075047
length_error: 0.895 token_f1: 0.176 rouge_l: 0.128
Exact Match Rate: 0.05112426035502959
length_error: 0.445 token_f1: 0.397 rouge_l: 0.28
Exact Match Rate: 0.1717557251908397
length_error: 0.365 token_f1: 0.609 rouge_l: 0.442
Exact Match Rate: 0.34220532319391633
length_error: 7.0 token_f1: 0.133 rouge_l: 0.095
Exact Match Rate: 0.2807017543859649
length_error: 0.907 token_f1: 0.148 rouge_l: 0.114
Exact Match Rate: 0.042045733956233095
length_error: 0.07 token_f1: 0.509 rouge_l: 0.305
Exact Match Rate: 0.25806451612903225
length_error: 0.251 token_f1: 0.362 rouge_l: 0.231
Exact Match Rate: 0.14948453608247422
length_error: 3.728 token_f1: 

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from metrics import compute_eval_metrics

baseline_data_A = get_baseline_data(dataset, target_model, "baseline")
baseline_data_B = get_baseline_data(dataset, target_model, "baseline_v0")

baseline_cots_A = [sample["ground_truth_cot"] for sample in baseline_data_A]
baseline_cots_B = [sample["ground_truth_cot"] for sample in baseline_data_B]

rouge_l_list = []
for baseline_c, tool_c in zip(baseline_cots_A, baseline_cots_B):
    if not tool_c:
        print("error")
        continue
    metrics = compute_eval_metrics(baseline_c, tool_c)
    length_error = metrics["length_error"]
    token_f1 = metrics["token_f1"]
    rouge_l = metrics["rouge_l"]
    rouge_l_list.append(rouge_l)
    print(length_error, rouge_l)

print(rouge_l_list)

1.427 0.28
0.605 0.241
0.187 0.243
0.399 0.266
0.215 0.24
0.09 0.333
0.25 0.429
0.347 0.183
[0.28, 0.241, 0.243, 0.266, 0.24, 0.333, 0.429, 0.183]


### REP Check

In [5]:
data_path = "../results/baseline_rep/glm-5.2/OpenThoughts/samples.jsonl"
records = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))
print(records[0].keys())

for idx in range(len(records)):
    if len(records[idx]["scratchpad_content"]) > 3:
        print("sample id:", records[idx]["sample_id"])
        print("target cot:", records[idx]["ground_truth_cot"])
        print("============="*20)
        print("scratchpad_content:", records[idx]["scratchpad_content"])
        # print("============="*20)
        # print("final_answer:", records[idx]["final_answer"])


dict_keys(['sample_id', 'source', 'question', 'ground_truth_cot', 'scratchpad_content', 'final_answer', 'token_count_ratio', 'length_error', 'token_f1', 'rouge_l', 'success'])
sample id: 91243
target cot: We need to solve: given strings s (length n) and t (length m), and q queries (l, r), count number of occurrences of t in substring s[l..r]. n,m up to 1000, q up to 1e5. So we can precompute a prefix sum array of counts of occurrences of t starting at each position in s, then answer queries quickly.

Specifically, for each i from 1 to n-m+1, check if s[i..i+m-1] == t. If yes, mark that an occurrence starts at i. Then build prefix sums: pref[i] = number of occurrences starting at positions <= i. For a query (l, r), occurrences fully within [l, r] are those that start at i where l <= i <= r-m+1 (and also i+m-1 <= r). So count = pref[r-m+1] - pref[l-1], provided r-m+1 >= l. If r-l+1 < m, answer 0. Need to handle boundaries.

Let's produce reasoning and final answer. Since this is a coding